In [ ]:
from langgraph.graph import StateGraph, START,END
from langchain_groq import ChatGroq
from typing import TypedDict,Literal
from dotenv import load_dotenv
from pydantic import BaseModel, Field
import os

In [ ]:
load_dotenv()

In [ ]:

model = ChatGroq(model="llama-3.1-8b-instant", 
               temperature=0.3,
                max_tokens=2048,
                api_key=os.getenv("GROQ_API_KEY"))

In [ ]:
class SentimentSchema(BaseModel):
    sentiment: Literal["Positive","Negative"]=Field(description="semtiment of the review")
    tone:Literal["angry","Frustrated","disappointed","Calm"]=Field(description="the emotional tone expressed by the user")
    urgency:Literal["low","Medium","High"]=Field(description="how urgent or critical the issue apperars to be ")
    

In [ ]:
class DiagnosisSchema(BaseModel):
    issue_type: Literal["UX","Performance","Bug","Support","Others"]
    tone: Literal["angry","Frustrated","disappointed","Calm"]
    urgency: Literal["low","Medium","High"]

In [ ]:
structured_model=model.with_structured_output(SentimentSchema)
structured_model2=model.with_structured_output(DiagnosisSchema)

In [ ]:
prompt="what is the sentiment of the followint review - the software is too bad "
structured_model.invoke(prompt)

In [ ]:
class ReviewState(TypedDict):
    review:str
    sentiment: Literal["Positive","Negative"]
    diagnosis: dict
    response: str

In [ ]:
def find_sentiment(state:ReviewState):
    prompt=f"for the following review, find out the sentiment \n {state['review']}"
    sentiment=structured_model.invoke(prompt).sentiment
    
    return {'sentiment':sentiment}



In [ ]:
def check_sentiment(state: ReviewState)-> Literal["positive_response","run_diagnosis"]:
    if state['sentiment']=="Positive":
        return "positive_response"
    else:
        return "run_diagnosis"


In [ ]:
def positive_response(state:ReviewState):
    prompt=f"""
Write a warm thank you message in response to this review:
\n\n"{state['review']}\"\n
Also, kindly ask the user to leave feedback on our Website"""
    response=model.invoke(prompt).content
    return {'response': response}



In [ ]:
def run_diagnosis(state: ReviewState):
    # diagnosis=state['diagnosis']
    prompt=f"""
Diagnose this negative review:

{state['review']}

Return:
- issue_type
- tone
- urgency
"""
    response=structured_model2.invoke(prompt)
    return {'diagnosis':response.model_dump()}

In [ ]:
def negative_response(state: ReviewState):
    diagnosis = state['diagnosis']
    prompt=f"""You are a support assistant.
    the user had a '{diagnosis['issue_type']}' issue, sounded '{diagnosis['tone']}', and marked urgency.
    Write an empatic, helpful resolution message.
"""
    response=model.invoke(prompt).content
    return {'response': response}

In [ ]:
graph=StateGraph(ReviewState)

graph.add_node('find_sentiment',find_sentiment)
graph.add_node('positive_response',positive_response)
graph.add_node('run_diagnosis',run_diagnosis)
graph.add_node('negative_response',negative_response)

In [ ]:
graph.add_edge(START,'find_sentiment')
graph.add_conditional_edges('find_sentiment',check_sentiment)
graph.add_edge('positive_response',END)
graph.add_edge('run_diagnosis','negative_response')
graph.add_edge('negative_response',END)


In [ ]:
workflow=graph.compile()

In [ ]:
workflow

In [ ]:
initial_state={
    'review':"""I have been using this app for about a month now, and i must say, the user intrface is Incredibly clean and intutive.
    Everything is exactly where you'd expect it to be. it's rare to find something that just work without needing a tutorial.
    Great job to the desighn team!"""
}
workflow.invoke(initial_state)

In [ ]:
initial_state={
    'review':"""I've been trying to log for over an hour now, and the app keeps freezing on authentication screen. I even tried 
    reinstalling it, but no luck. this kind of bug is unaccetable, especially when it affect the basic fiunctionality  """
}
workflow.invoke(initial_state)